# Feature Importance Analysis

Analyze feature importance across trained models to understand:
- Which features are universally important
- Which features are target-specific
- Feature redundancy and correlation
- Latent knowledge feature performance

Goal: Guide future feature engineering and data acquisition.

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict

from src.ml import ModelRegistry, TrainedModel
from src.data.feature_engineering import (
    EnhancedTechnicalFeatures,
    AlternativeDataFeatures,
    LatentKnowledgeFeatures,
)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Feature Importance Data

In [ ]:
# Load all trained models
registry = ModelRegistry(Path("~/quant_results/models").expanduser())
registry.load_all_from_disk()

# Collect feature importance from all models
all_importances = defaultdict(list)
model_importances = {}

for model_id in registry.list_models(validated_only=False):
    try:
        model = registry.load(model_id)
        
        # Get feature importance
        if hasattr(model.model, 'feature_importances_'):
            importances = dict(zip(model.feature_columns, model.model.feature_importances_))
        elif hasattr(model.model, 'coef_'):
            importances = dict(zip(model.feature_columns, np.abs(model.model.coef_).flatten()))
        else:
            continue
        
        model_importances[model_id] = {
            'importances': importances,
            'target': model.target_config.name,
            'model_type': model.config.model_type.value,
        }
        
        # Aggregate across models
        for feature, importance in importances.items():
            all_importances[feature].append(importance)
    
    except Exception as e:
        print(f"Error loading {model_id}: {e}")

print(f"Loaded importance data from {len(model_importances)} models")
print(f"Total unique features: {len(all_importances)}")

## 2. Universal Feature Importance

In [ ]:
# Compute average importance across all models
if all_importances:
    avg_importance = {
        feature: np.mean(values)
        for feature, values in all_importances.items()
    }
    
    importance_df = pd.DataFrame([
        {
            'feature': feature,
            'mean_importance': np.mean(values),
            'std_importance': np.std(values),
            'n_models': len(values),
            'consistency': np.mean(values) / (np.std(values) + 1e-10),  # Mean/std ratio
        }
        for feature, values in all_importances.items()
    ]).sort_values('mean_importance', ascending=False)
    
    print("Top 20 Most Important Features (across all models):")
    display(importance_df.head(20))

In [ ]:
# Visualize top features
if len(importance_df) > 0:
    top_n = min(30, len(importance_df))
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    top_features = importance_df.head(top_n)
    
    ax.barh(range(top_n), top_features['mean_importance'], 
            xerr=top_features['std_importance'], capsize=3)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top_features['feature'])
    ax.invert_yaxis()
    ax.set_xlabel('Mean Importance')
    ax.set_title(f'Top {top_n} Features by Importance')
    
    plt.tight_layout()
    plt.show()

## 3. Feature Importance by Category

In [ ]:
# Define feature categories
def categorize_feature(name):
    """Categorize feature by name pattern."""
    name = name.lower()
    
    if any(x in name for x in ['rsi', 'macd', 'bb_', 'sma', 'ema', 'momentum', 'volatility']):
        return 'Technical'
    elif any(x in name for x in ['congress', 'insider', 'legislative']):
        return 'Congressional/Insider'
    elif any(x in name for x in ['sentiment', 'aaii', 'newsletter', 'social']):
        return 'Sentiment'
    elif any(x in name for x in ['put_call', 'options', 'iv_', 'unusual']):
        return 'Options Flow'
    elif any(x in name for x in ['gap', 'monday', 'calendar', 'month_end', 'opex']):
        return 'Calendar/Seasonality'
    elif any(x in name for x in ['vix', 'vol_regime', 'rates', 'dollar', 'credit']):
        return 'Macro/Regime'
    elif any(x in name for x in ['earnings', 'fda', 'event']):
        return 'Events'
    elif any(x in name for x in ['sector', 'lead', 'correlation']):
        return 'Cross-Asset'
    else:
        return 'Other'

if len(importance_df) > 0:
    importance_df['category'] = importance_df['feature'].apply(categorize_feature)
    
    category_summary = importance_df.groupby('category').agg({
        'feature': 'count',
        'mean_importance': 'mean',
        'consistency': 'mean',
    }).rename(columns={'feature': 'n_features'})
    
    print("Feature Importance by Category:")
    display(category_summary.sort_values('mean_importance', ascending=False))

In [ ]:
# Box plot by category
if len(importance_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    importance_df.boxplot(column='mean_importance', by='category', ax=ax, rot=45)
    ax.set_title('Feature Importance Distribution by Category')
    ax.set_xlabel('')
    ax.set_ylabel('Mean Importance')
    plt.suptitle('')
    
    plt.tight_layout()
    plt.show()

## 4. Latent Knowledge Features Analysis

In [ ]:
# Focus on latent knowledge features
latent_features = LatentKnowledgeFeatures.get_feature_list()
print(f"Latent Knowledge Features: {len(latent_features)}")

if len(importance_df) > 0:
    latent_importance = importance_df[importance_df['feature'].isin(latent_features)]
    
    print(f"\nLatent features with importance data: {len(latent_importance)}")
    
    if len(latent_importance) > 0:
        print("\nLatent Knowledge Feature Importance:")
        display(latent_importance.sort_values('mean_importance', ascending=False))

In [ ]:
# Latent knowledge hypotheses validation
hypotheses = LatentKnowledgeFeatures.get_hypotheses()

print("Latent Knowledge Hypotheses Status:")
print("=" * 60)

for name, description in hypotheses.items():
    # Check if any related features are in top 50% by importance
    if len(importance_df) > 0:
        median_importance = importance_df['mean_importance'].median()
        related = importance_df[importance_df['feature'].str.contains(name.replace('_', '|'))]
        if len(related) > 0:
            avg_imp = related['mean_importance'].mean()
            status = '✓ Validated' if avg_imp > median_importance else '? Weak'
        else:
            status = '- Not tested'
    else:
        status = '- No data'
    
    print(f"\n{name}:")
    print(f"  {description}")
    print(f"  Status: {status}")

## 5. Feature Importance by Target

In [ ]:
# Aggregate importance by target
if model_importances:
    target_importances = defaultdict(lambda: defaultdict(list))
    
    for model_id, data in model_importances.items():
        target = data['target']
        for feature, importance in data['importances'].items():
            target_importances[target][feature].append(importance)
    
    # Build comparison table for top features
    if len(importance_df) > 0:
        top_features_list = importance_df.head(20)['feature'].tolist()
        
        comparison_data = []
        for feature in top_features_list:
            row = {'feature': feature}
            for target in target_importances.keys():
                values = target_importances[target].get(feature, [])
                row[target] = np.mean(values) if values else 0
            comparison_data.append(row)
        
        target_comparison = pd.DataFrame(comparison_data).set_index('feature')
        
        print("Top 20 Features Importance by Target:")
        display(target_comparison.style.background_gradient(cmap='YlOrRd', axis=None))

## 6. Feature Consistency Analysis

In [ ]:
# Analyze feature stability across models
if len(importance_df) > 0:
    # Most consistent features (high importance, low variance)
    consistent_features = importance_df[
        (importance_df['n_models'] >= 3) &
        (importance_df['consistency'] > 1)
    ].sort_values('mean_importance', ascending=False)
    
    print("Most Consistent High-Importance Features:")
    display(consistent_features.head(15))

In [ ]:
# Scatter: importance vs consistency
if len(importance_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 8))
    
    scatter = ax.scatter(
        importance_df['mean_importance'],
        importance_df['consistency'],
        c=importance_df['n_models'],
        cmap='viridis',
        alpha=0.7,
        s=50,
    )
    
    plt.colorbar(scatter, label='# Models')
    
    # Label top features
    for _, row in importance_df.head(10).iterrows():
        ax.annotate(
            row['feature'],
            (row['mean_importance'], row['consistency']),
            fontsize=8,
        )
    
    ax.set_xlabel('Mean Importance')
    ax.set_ylabel('Consistency (Mean/Std)')
    ax.set_title('Feature Importance vs Consistency')
    
    plt.tight_layout()
    plt.show()

## 7. Recommendations

In [ ]:
# Generate recommendations
print("=== Feature Importance Analysis Recommendations ===")

if len(importance_df) > 0:
    print("\n1. KEEP (High importance, consistent):")
    keep = importance_df[
        (importance_df['mean_importance'] > importance_df['mean_importance'].quantile(0.8)) &
        (importance_df['consistency'] > 1)
    ]['feature'].tolist()[:10]
    for f in keep:
        print(f"   - {f}")
    
    print("\n2. INVESTIGATE (High importance, inconsistent):")
    investigate = importance_df[
        (importance_df['mean_importance'] > importance_df['mean_importance'].quantile(0.7)) &
        (importance_df['consistency'] < 0.5)
    ]['feature'].tolist()[:5]
    for f in investigate:
        print(f"   - {f}")
    
    print("\n3. CONSIDER REMOVING (Low importance):")
    remove = importance_df[
        importance_df['mean_importance'] < importance_df['mean_importance'].quantile(0.2)
    ]['feature'].tolist()[:5]
    for f in remove:
        print(f"   - {f}")
    
    print("\n4. Feature Categories Ranking:")
    if 'category' in importance_df.columns:
        category_rank = category_summary.sort_values('mean_importance', ascending=False)
        for i, (cat, row) in enumerate(category_rank.iterrows(), 1):
            print(f"   {i}. {cat}: {row['mean_importance']:.4f} avg importance")
else:
    print("No importance data available. Train models first.")